In [ ]:
# Inport Required Libraries
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [9]:
# Upload Data set
df=pd.read_csv('breast_cancer.csv')
df.head(4)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300


In [10]:
# Drop the Id column
df=df.drop(columns='id')
df.head(2)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902


In [11]:
# Extract Input and Oputput Value
X=df.iloc[:, 1:]
y=df.iloc[:, 0]

In [13]:
# Split the dataset into train and test set
X_train, X_test, y_train, y_test=train_test_split(
    X,y, test_size=0.2
)

In [14]:
# Scale the train data
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [15]:
# Perform Label Encoding on Target Column
encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

In [16]:
# Convert numpy array to pytorch tensor
X_train_tensor=torch.from_numpy(X_train_scaled)
X_test_tensor=torch.from_numpy(X_test_scaled)
y_train_tensor=torch.from_numpy(y_train)
y_test_tensor=torch.from_numpy(y_test)


In [23]:
# Define the model
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss

In [19]:
# Define Important parameter
learning_rate=0.1
epochs=25

In [24]:
# create model
model = MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 3.0936807712257757
Epoch: 2, Loss: 2.9730025173710697
Epoch: 3, Loss: 2.8530623884464412
Epoch: 4, Loss: 2.7356933875846434
Epoch: 5, Loss: 2.617212136253672
Epoch: 6, Loss: 2.492212127010902
Epoch: 7, Loss: 2.3674141414710244
Epoch: 8, Loss: 2.2450684146141846
Epoch: 9, Loss: 2.1272106453144404
Epoch: 10, Loss: 2.010576359318573
Epoch: 11, Loss: 1.8922425847429705
Epoch: 12, Loss: 1.7738453602698818
Epoch: 13, Loss: 1.6627026590658858
Epoch: 14, Loss: 1.5593870198273212
Epoch: 15, Loss: 1.4642877270068435
Epoch: 16, Loss: 1.3726204457229656
Epoch: 17, Loss: 1.2856636808587851
Epoch: 18, Loss: 1.2081532288114076
Epoch: 19, Loss: 1.1391803379895233
Epoch: 20, Loss: 1.080088663238353
Epoch: 21, Loss: 1.030068939385268
Epoch: 22, Loss: 0.9880879876664643
Epoch: 23, Loss: 0.9530032744726148
Epoch: 24, Loss: 0.9236840129591208
Epoch: 25, Loss: 0.8990925734408445


In [27]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5734071731567383
